## 1. Evaluation Metrics For ASR
* metrics for assessing speech recognition systems will be similar to **Levenshtein distance**.
* We categorise these errors from an ASR system into one of three categories:
  1. **Substitutions (S):** where we transcribe the wrong word in our prediction (“sit” instead of “sat”)
  2. **Insertions (I):** where we add an extra word in our prediction
  3. **Deletions (D):** where we remove a word in our prediction

* The level at which we compute these errors: we can either compute them on the *word level* or on the *character level*.

## 2. Word Error Rate

* The **word error rate (WER)** metric calculates substitutions, insertions and deletions on the *word level*.
* Example:
  ```python
  reference = "the cat sat on the mat"
  prediction = "the cat sit on the"
  ```
  | Reference:  | the | cat | sat | on | the | mat |
  |-------------|-----|-----|-----|----|-----|-----|
  | Prediction: | the | cat | sit | on | the |     |
  | Label:      | ✅  | ✅  | S   | ✅  | ✅  | D   |

  * Here, we have:
    * 1 substitution (“sit” instead of “sat”)
    * 0 insertions
    * 1 deletion (“mat” is missing)
  * This gives 2 errors in total. To get our error rate, we divide the number of errors by the total number of words in our reference (N), which for this example is 6:
  \begin{align}
  WER &= \frac{S + I + D}{N} \\
  &= \frac{1 + 0 + 1}{6} \\
  &= 0.333
  \end{align}
  where:
    * S = substitutions
    * D = deletions
    * I = insertions
    * N = number of words in the reference transcript

* The WER is defined such that *lower is better*: a lower WER means there are fewer errors in our prediction, so a perfect speech recognition system would have a WER of zero (no errors).



In [5]:
!pip install --upgrade evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.1 MB/s eta 0:00:00


In [3]:
# Load up the WER metric and compute it for our above example

from evaluate import load

wer_metric = load("wer")

reference = "the cat sat on the mat"
prediction = "the cat sit on the"

wer = wer_metric.compute(references=[reference], predictions=[prediction])
print(wer)

0.3333333333333333


## 3. Inverse Real-Time Factor (RTFx)
* The inverse real-time factor (RTFx) measures the **speed** of an ASR system.
* RTFx is the inverse ratio of processing time to audio duration:
  \begin{equation}
  \text{RTFx} = \frac{\text{Audio Duration}}{\text{Processing Time}}
  \end{equation}
* For example, if it takes 10 seconds to transcribe 100 seconds of audio, the RTFx is 100/10 = 10. An RTFx greater than 1.0 means the system can transcribe audio faster than real-time, which is essential for live transcription applications like video conferencing or live captioning.
* Key points about RTFx:
  * Higher is better: Higher RTFx means faster processing
  * RTFx > 1.0: Faster than real-time (good for streaming applications)
  * RTFx = 1.0: Processes at exactly real-time speed
  * RTFx < 1.0: Slower than real-time (may be acceptable for batch processing)
* RTFx is hardware-dependent and varies based on factors like:
  * Model size (larger models typically have lower RTFx)
  * Hardware acceleration (GPU vs CPU)
  * Batch size
  * Audio characteristics (sampling rate, number of channels)

## 4. Normalisation
* If we train an ASR model on data with punctuation and casing, it will learn to predict casing and punctuation in its transcriptions -> a style referred to as *orthographic*.
* **Normalising** means removing casing and punctuation from the texts before evaluation (applied to both reference and prediction).
* Normalising the dataset makes the speech recognition task easier: the model no longer needs to distinguish between upper and lower case characters, or have to predict punctuation from the audio data alone.
  * While we get lower WERs, the model isn’t necessarily better for production.
  * The lack of casing and punctuation makes the predicted text from the model significantly harder to read.
* The Whisper transcription is orthographic and thus ready to go. While, Wav2Vec2 model predicts neither punctuation nor casing.
* There is a happy medium between normalising and not normalising: we can train our systems on orthographic transcriptions, and then normalise the predictions and targets before computing the WER.
* The Whisper model was released with a normaliser that effectively handles the normalisation of casing, punctuation and number formatting among others.

In [4]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

prediction = " He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similarly is drawn from eating and its results occur most readily to the mind."
normalized_prediction = normalizer(prediction)

normalized_prediction

' he tells us that at this festive season of the year with christmas and roast beef looming before us similarly is drawn from eating and its results occur most readily to the mind '

In [5]:
reference = "HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND"
normalized_referece = normalizer(reference)

wer = wer_metric.compute(
    references=[normalized_referece], predictions=[normalized_prediction]
)
wer

0.0625

# Putting It All Together
1. Pre-trained models
2. Dataset selection
3. Evaluation


In [1]:
import torch

# Set up the GPU
if torch.cuda.is_available():
    device = "cuda:0"
    # load the model in half-precision (float16) if running on a GPU
    # this will speed up inference at almost no cost to WER accuracy.
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
# check what process we use:
print(f"device: {device}, torch_dtype: {torch_dtype}")

device: cuda:0, torch_dtype: torch.float16


In [2]:
# Grant the huggingface token
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [3]:
# Load the dataset: gigaspeech
# GigaSpeech is an evolving, multi-domain English speech recognition corpus

from datasets import load_dataset
from datasets import Audio

gigaspeech = load_dataset("speechcolab/gigaspeech", "xs", streaming=True)
sampling_rate = gigaspeech["test"].features["audio"].sampling_rate

# see structure
print(gigaspeech)

# take only 10 samples from test
gigaspeech_test_10 = gigaspeech["test"].take(10)
gigaspeech_test_10 = gigaspeech_test_10.cast_column("audio", Audio(sampling_rate=16000))

num_audios = sum(1 for _ in gigaspeech_test_10)
print(f"Number of Audios: {num_audios}")

README.md: 0.00B [00:00, ?B/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['segment_id', 'speaker', 'text', 'audio', 'begin_time', 'end_time', 'audio_id', 'title', 'url', 'source', 'category', 'original_full_path'],
        num_shards: 1
    })
    validation: IterableDataset({
        features: ['segment_id', 'speaker', 'text', 'audio', 'begin_time', 'end_time', 'audio_id', 'title', 'url', 'source', 'category', 'original_full_path'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['segment_id', 'speaker', 'text', 'audio', 'begin_time', 'end_time', 'audio_id', 'title', 'url', 'source', 'category', 'original_full_path'],
        num_shards: 2
    })
})
Number of Audios: 10


In [4]:
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.to(device)
model.eval()

all_predictions = []
all_references = []

for sample in tqdm(gigaspeech_test_10, total=10):
    audio_array = sample["audio"]["array"]
    sampling_rate = sample["audio"]["sampling_rate"]

    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(**inputs, task="transcribe")

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    all_predictions.append(transcription)
    all_references.append(sample["text"])

    print(f"{transcription}")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

 These babies probably aren't part of your typical Friday night dinner. There are many varieties of this legendary delicacy, but just one can be named the literal king of all loisters.


 90%|█████████ | 9/10 [00:21<00:01,  1.19s/it]

 you


100%|██████████| 10/10 [00:21<00:00,  2.20s/it]

 That said, astronomy is facing a total wave of data.


In [8]:
# Evaluation
from evaluate import load

wer_metric = load("wer")

wer_ortho = 100 * wer_metric.compute(
    references=all_references,
    predictions=all_predictions
)

print(f"WER Orthographic: {wer_ortho:.2f}%")

WER Orthographic: 105.93%


In [9]:
# Evaluate the normalised WER
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

# compute normalised WER
all_predictions_norm = [normalizer(pred) for pred in all_predictions]
all_references_norm = [normalizer(label) for label in all_references]

# filtering step to only evaluate the samples that correspond to non-zero references
all_predictions_norm = [
    all_predictions_norm[i]
    for i in range(len(all_predictions_norm))
    if len(all_references_norm[i]) > 0
]
all_references_norm = [
    all_references_norm[i]
    for i in range(len(all_references_norm))
    if len(all_references_norm[i]) > 0
]

wer = 100 * wer_metric.compute(
    references=all_references_norm,
    predictions=all_predictions_norm
)

print(f"WER: {wer:.2f}%")

WER: 10.20%
